Weapons and Ammunition data analysis


In [5]:
import pandas as pd
pd.options.display.max_columns = None

df = pd.read_csv('data/amcdata_weapons_facilities_V2.csv', encoding='latin-1')
df = df[df['summary_category'] != 1]
other_df = df[df['item_type'] == 3]

### Select weapons life cycle stages

In [6]:
cols_to_keep = [
    # Identifiers
    'item_type', 'item',

    # Lifecycle stages - ban flags
    'ban_development',
    'ban_testing',
    'ban_production',
    'ban_acquisition',
    'ban_possession',
    'ban_station',
    'ban_transfer',
    'ban_use',
    'ban_disposal',

    # Lifecycle stages - restriction flags
    'restriction_development',
    'testing_restriction',       # note: inconsistent naming in the dataset
    'restriction_production',
    'restriction_acquisition',
    'restriction_possession',
    'restriction_transfer',
    'restriction_use',
    'restriction_disposal',

    # End-of-life stages (no ban/restriction equivalents)
    'eliminitation',             # note: typo in the dataset
    'conversion',
    'modernization',
    'facility_destruction',
]

other_lifecycle_df = other_df[cols_to_keep]


Which columns are causing problems due to no variance?

In [7]:
ban_cols = [c for c in other_lifecycle_df.columns if c.startswith('ban_')]
restriction_cols = [c for c in other_lifecycle_df.columns if 'restriction' in c]

flag_cols = ban_cols + restriction_cols

for col in flag_cols:
    unique_vals = other_lifecycle_df[col].dropna().unique()
    if len(unique_vals) <= 1:
        print(f"{col}: only contains {unique_vals}")

ban_development: only contains [0]
ban_testing: only contains [0]
ban_possession: only contains [0]
ban_station: only contains [0]


### Testing correlations between different life cycle stages of weapons/ammunitions

In [8]:
# Separate ban and restriction columns
ban_cols = [c for c in other_lifecycle_df.columns if c.startswith('ban_')]
restriction_cols = [c for c in other_lifecycle_df.columns if 'restriction' in c]

# Correlation between every ban col vs every restriction col
corr_matrix = other_lifecycle_df[ban_cols + restriction_cols].corr()

# Slice to only show ban vs restriction (not ban vs ban or restriction vs restriction)
corr_ban_vs_restriction = corr_matrix.loc[ban_cols, restriction_cols]
print(corr_ban_vs_restriction)

                 restriction_development  testing_restriction  \
ban_development                      NaN                  NaN   
ban_testing                          NaN                  NaN   
ban_production                 -0.053376            -0.053376   
ban_acquisition                -0.053376            -0.053376   
ban_possession                       NaN                  NaN   
ban_station                          NaN                  NaN   
ban_transfer                   -0.096077            -0.096077   
ban_use                        -0.053376            -0.053376   
ban_disposal                   -0.160128            -0.160128   

                 restriction_production  restriction_acquisition  \
ban_development                     NaN                      NaN   
ban_testing                         NaN                      NaN   
ban_production                -0.111111                -0.100504   
ban_acquisition               -0.111111                -0.100504   
ban_posse

In [9]:
# Stack matrix into a series and sort
corr_ranked = (
    corr_ban_vs_restriction
    .stack()
    .reset_index()
    .rename(columns={'level_0': 'ban', 'level_1': 'restriction', 0: 'correlation'})
    .sort_values('correlation', ascending=False)
)

print(corr_ranked.head(40))

                ban              restriction  correlation
7    ban_production     restriction_disposal    -0.037037
15  ban_acquisition     restriction_disposal    -0.037037
31          ban_use     restriction_disposal    -0.037037
0    ban_production  restriction_development    -0.053376
8   ban_acquisition  restriction_development    -0.053376
9   ban_acquisition      testing_restriction    -0.053376
1    ban_production      testing_restriction    -0.053376
24          ban_use  restriction_development    -0.053376
25          ban_use      testing_restriction    -0.053376
23     ban_transfer     restriction_disposal    -0.066667
17     ban_transfer      testing_restriction    -0.096077
16     ban_transfer  restriction_development    -0.096077
3    ban_production  restriction_acquisition    -0.100504
27          ban_use  restriction_acquisition    -0.100504
11  ban_acquisition  restriction_acquisition    -0.100504
2    ban_production   restriction_production    -0.111111
10  ban_acquis

In [10]:
# Pearson - default, fine for binary
other_lifecycle_df[ban_cols + restriction_cols].corr(method='pearson')

,ban_development,ban_testing,ban_production,ban_acquisition,ban_possession,ban_station,ban_transfer,ban_use,ban_disposal,restriction_development,testing_restriction,restriction_production,restriction_acquisition,restriction_possession,restriction_transfer,restriction_use,restriction_disposal
ban_development,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ban_testing,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ban_production,NaN,NaN,1.000000,1.000000,NaN,NaN,-0.066667,1.000000,-0.111111,-0.053376,-0.053376,-0.111111,-0.100504,-0.111111,-0.192450,-0.154807,-0.037037
ban_acquisition,NaN,NaN,1.000000,1.000000,NaN,NaN,-0.066667,1.000000,-0.111111,-0.053376,-0.053376,-0.111111,-0.100504,-0.111111,-0.192450,-0.154807,-0.037037
ban_possession,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ban_station,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ban_transfer,NaN,NaN,-0.066667,-0.066667,NaN,NaN,1.000000,-0.066667,-0.200000,-0.096077,-0.096077,-0.200000,-0.180907,-0.200000,-0.346410,-0.278652,-0.066667
ban_use,NaN,NaN,1.000000,1.000000,NaN,NaN,-0.066667,1.000000,-0.111111,-0.053376,-0.053376,-0.111111,-0.100504,-0.111111,-0.192450,-0.154807,-0.037037
ban_disposal,NaN,NaN,-0.111111,-0.111111,NaN,NaN,-0.200000,-0.111111,1.000000,-0.160128,-0.160128,-0.333333,-0.301511,-0.333333,-0.412393,-0.295540,-0.111111
restriction_development,NaN,NaN,-0.053376,-0.053376,NaN,NaN,-0.096077,-0.053376,-0.160128,1.000000,1.000000,0.480384,-0.144841,0.480384,0.277350,-0.223100,-0.053376


In [11]:
# Spearman - better for ordinal/binary data, more robust
other_lifecycle_df[ban_cols + restriction_cols].corr(method='spearman')

,ban_development,ban_testing,ban_production,ban_acquisition,ban_possession,ban_station,ban_transfer,ban_use,ban_disposal,restriction_development,testing_restriction,restriction_production,restriction_acquisition,restriction_possession,restriction_transfer,restriction_use,restriction_disposal
ban_development,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ban_testing,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ban_production,NaN,NaN,1.000000,1.000000,NaN,NaN,-0.066667,1.000000,-0.111111,-0.053376,-0.053376,-0.111111,-0.100504,-0.111111,-0.192450,-0.154807,-0.037037
ban_acquisition,NaN,NaN,1.000000,1.000000,NaN,NaN,-0.066667,1.000000,-0.111111,-0.053376,-0.053376,-0.111111,-0.100504,-0.111111,-0.192450,-0.154807,-0.037037
ban_possession,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ban_station,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ban_transfer,NaN,NaN,-0.066667,-0.066667,NaN,NaN,1.000000,-0.066667,-0.200000,-0.096077,-0.096077,-0.200000,-0.180907,-0.200000,-0.346410,-0.278652,-0.066667
ban_use,NaN,NaN,1.000000,1.000000,NaN,NaN,-0.066667,1.000000,-0.111111,-0.053376,-0.053376,-0.111111,-0.100504,-0.111111,-0.192450,-0.154807,-0.037037
ban_disposal,NaN,NaN,-0.111111,-0.111111,NaN,NaN,-0.200000,-0.111111,1.000000,-0.160128,-0.160128,-0.333333,-0.301511,-0.333333,-0.412393,-0.295540,-0.111111
restriction_development,NaN,NaN,-0.053376,-0.053376,NaN,NaN,-0.096077,-0.053376,-0.160128,1.000000,1.000000,0.480384,-0.144841,0.480384,0.277350,-0.223100,-0.053376
